In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib ipympl

from ipywidgets import widgets
from ipywidgets.widgets import fixed
from topepan import plot as tpp
import numpy as np

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
# The folder holding the .tec output. Edit this line and re-run from here down; it is the one
# thing that changes between runs, which is why it lives in the notebook rather than in a config
# file you would have to open separately.
RUN_FOLDER = '/path/to/your/crunchtope/run/'

catList, max_time = tpp.data_cats(RUN_FOLDER)

In [ ]:
file_cat = widgets.ToggleButtons(
    options=catList,
    description='Output type',
    button_style='')
time_slider = widgets.IntSlider(value=1, min=1, max=max_time, step=1,
                                description='Time step')
plot_var = widgets.Select(options=(tpp.read_tecplot(file_cat.value, time_slider.value)[1])[3:],
                          description='Variable')
# X down the y axis is the depth convention and how this browser has always drawn; X across the
# x axis reads better for a flow path. Both are useful for a 1-D column, so pick per plot.
orientation = widgets.ToggleButtons(
    options=[('X on y axis (depth)', True), ('X on x axis', False)],
    value=True,
    description='Orientation')


def update_plot_vars(*args):
    plot_var.options = tpp.read_tecplot(file_cat.value, time_slider.value)[1][3:]


def update_plot(time, file_cat, plot_var, orientation):
    if plot_var is None:
        return
    df, column_headers = tpp.read_tecplot(file_cat, time)
    # Ranged over every timestep, so the axis does not jump about as the slider moves.
    lower, upper = tpp.plot_var_range(time_slider.max, file_cat, plot_var)

    # Pad the value axis away from the data, and give a flat profile a range it can be seen in.
    if upper > 0:
        upper = upper * 1.02
    elif upper < 0:
        upper = upper * 0.98
    else:
        upper = -lower

    if lower > 0:
        lower = lower * 0.98
    elif lower < 0:
        lower = lower * 1.02
    else:
        lower = -upper

    if upper == lower:
        upper = upper + 1
        lower = lower - 1

    tpp.draw_profile(ax, df['X'], df[plot_var], vertical=orientation,
                     lower=lower, upper=upper, value_label=plot_var)
    fig.canvas.draw_idle()


file_cat.observe(update_plot_vars, 'value')
fig, ax, line = tpp.initialise1D('totcon')
widgets.interact(update_plot, file_cat=file_cat, time=time_slider, plot_var=plot_var,
                 orientation=orientation)

In [ ]:
plt.close('all')